In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import linprog

freight = pd.read_csv('freight_rates.csv', parse_dates=['date'])
bunker = pd.read_csv('bunker_prices.csv', parse_dates=['date'])
port_calls = pd.read_csv('port_calls.csv', parse_dates=['arrival_time','berth_time','departure_time'])
vessels = pd.read_csv('vessels.csv')

port_coords_data = [
    ("Gladstone","Australia",-23.8489,151.2555), ("Newcastle","Australia",-32.9283,151.7817),
    ("Hay Point","Australia",-21.2833,149.3000), ("Dalrymple Bay","Australia",-21.2667,149.2833),
    ("Taboneo","Indonesia",-3.4500,114.6167), ("Muara Pantai","Indonesia",-0.4500,117.1167),
    ("Samarinda","Indonesia",-0.5022,117.1536), ("Hampton Roads","USA",36.9468,-76.3300),
    ("Baltimore","USA",39.2667,-76.5833), ("New Orleans","USA",29.9500,-90.0667),
    ("Beira","Mozambique",-19.8433,34.8383), ("Nacala","Mozambique",-14.5658,40.6725),
    ("Ust-Luga (Baltic)","Russia",59.6667,28.3500), ("Novorossiysk (Black Sea)","Russia",44.7239,37.7683),
    ("Vostochny (Far East)","Russia",42.7500,133.0833), ("Dhamra","India",20.7833,86.9167),
    ("Gangavaram","India",17.6167,83.2333), ("Gopalpur","India",19.2667,84.9167),
    ("Haldia","India",22.0333,88.0667), ("Paradip","India",20.3167,86.6167),
    ("Vizag","India",17.6868,83.2185),
]
coords = pd.DataFrame(port_coords_data, columns=["port_name","country","latitude","longitude"]).set_index('port_name')

def haversine_nm(lat1, lon1, lat2, lon2):
    R = 3440.065
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2-lat1; dlon = lon2-lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

In [2]:
ORIGIN_PORT, ORIGIN_CODE = "Gladstone", "AUS"
DEST_PORT, DEST_CODE = "Paradip", "PAR"
VESSEL_CLASS = "Panamax"
TOTAL_CARGO_MT = 480_000
PERIOD_MONTHS = 6

route_id = f"{ORIGIN_CODE}_{DEST_CODE}_{'PAN' if VESSEL_CLASS=='Panamax' else 'CAP'}"

In [3]:
v = vessels[vessels['vessel_type']==VESSEL_CLASS].iloc[0]
cargo_per_voyage = v['dwt'] * 0.92
distance_nm = haversine_nm(coords.loc[ORIGIN_PORT,'latitude'], coords.loc[ORIGIN_PORT,'longitude'],
                             coords.loc[DEST_PORT,'latitude'], coords.loc[DEST_PORT,'longitude'])
laden_days = distance_nm / (v['speed_laden'] * 24)
ballast_days = distance_nm / (v['speed_ballast'] * 24)

route_freight = freight[freight['route_id']==route_id]
avg_spot_rate = route_freight['freight_usd_mt'].mean()
avg_bunker_price = bunker[bunker['fuel_type']=='VLSFO']['price'].mean()
avg_wait_hours = port_calls[port_calls['port_id']==DEST_CODE]['waiting_hours'].mean()
congestion_days = avg_wait_hours / 24

n_voyages_needed = int(np.ceil(TOTAL_CARGO_MT / cargo_per_voyage))
print(f"Distance: {distance_nm:.0f} nm | Voyages needed: {n_voyages_needed}")
print(f"Avg spot rate: ${avg_spot_rate:.2f}/mt | Avg bunker: ${avg_bunker_price:.2f}/mt")

Distance: 4616 nm | Voyages needed: 7
Avg spot rate: $16.64/mt | Avg bunker: $524.75/mt


In [4]:
bunker_tons = v['fuel_consumption_laden']*laden_days + v['fuel_consumption_ballast']*ballast_days
bunker_cost_per_voyage = bunker_tons * avg_bunker_price
voyage_days = laden_days + ballast_days
congestion_cost_per_voyage = congestion_days * (avg_spot_rate * cargo_per_voyage / voyage_days)
idle_cost_per_voyage = 1.5 * (avg_spot_rate * cargo_per_voyage / voyage_days)   # proxy: ~1.5 idle days/voyage
deadhead_cost_per_voyage = v['fuel_consumption_ballast'] * ballast_days * avg_bunker_price
risk_penalty_per_voyage = 0.02 * avg_spot_rate * cargo_per_voyage

fixed_cost_per_voyage = (bunker_cost_per_voyage + congestion_cost_per_voyage +
                          idle_cost_per_voyage + deadhead_cost_per_voyage + risk_penalty_per_voyage)
print(f"Fixed cost per voyage (everything except freight): ${fixed_cost_per_voyage:,.0f}")

Fixed cost per voyage (everything except freight): $650,709


In [5]:
contract_types = {
    "Spot":              {"discount": 0.00, "max_voyages": n_voyages_needed},
    "3-voyage contract": {"discount": 0.03, "max_voyages": 3},
    "6-voyage COA":       {"discount": 0.06, "max_voyages": 6},
    "12-voyage COA":      {"discount": 0.09, "max_voyages": 12},
}
names = list(contract_types.keys())
freight_cost_per_voyage = {name: avg_spot_rate*(1-c["discount"])*cargo_per_voyage for name, c in contract_types.items()}
total_cost_per_voyage = {name: freight_cost_per_voyage[name] + fixed_cost_per_voyage for name in names}

for name in names:
    print(f"{name}: ${total_cost_per_voyage[name]:,.0f} per voyage")

Spot: $1,902,094 per voyage
3-voyage contract: $1,864,552 per voyage
6-voyage COA: $1,827,011 per voyage
12-voyage COA: $1,789,469 per voyage


In [6]:
c = [total_cost_per_voyage[name] for name in names]
A_ub = [[-1]*len(names)]
b_ub = [-n_voyages_needed]

max_share = 0.5   # no single contract type covers more than 50% of voyages (risk diversification)
bounds = [(0, min(contract_types[name]['max_voyages'], n_voyages_needed*max_share)) for name in names]

res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')

print("--- OPTIMIZER RESULT ---")
for name, v_ in zip(names, res.x):
    print(f"  {name}: {v_:.1f} voyages ({v_/n_voyages_needed*100:.0f}%)")

current_plan_cost = total_cost_per_voyage["Spot"] * n_voyages_needed
optimal_cost = res.fun
print(f"\nCurrent plan (100% Spot): ${current_plan_cost:,.0f}")
print(f"Optimized cost: ${optimal_cost:,.0f}")
print(f"Expected saving: ${current_plan_cost - optimal_cost:,.0f}")

--- OPTIMIZER RESULT ---
  Spot: 0.0 voyages (0%)
  3-voyage contract: 0.0 voyages (0%)
  6-voyage COA: 3.5 voyages (50%)
  12-voyage COA: 3.5 voyages (50%)

Current plan (100% Spot): $13,314,658
Optimized cost: $12,657,681
Expected saving: $656,977


In [7]:
def run_optimizer(origin_port, origin_code, dest_port, dest_code, vessel_class, total_cargo_mt, max_share=0.5):
    # (paste the body of Steps 3-6 here, replacing the hardcoded variables with the function args)
    ...
    return {"n_voyages": n_voyages_needed, "current_plan_cost": current_plan_cost,
            "optimal_cost": optimal_cost, "saving": current_plan_cost - optimal_cost,
            "mix": dict(zip(names, res.x))}

# manager changes cargo quantity, origin, destination, date -- Phase 38's exact ask
run_optimizer("Vostochny (Far East)", "RUS", "Paradip", "PAR", "Panamax", 300_000)

{'n_voyages': 7,
 'current_plan_cost': np.float64(13314658.088858051),
 'optimal_cost': 12657681.044229716,
 'saving': np.float64(656977.0446283352),
 'mix': {'Spot': np.float64(0.0),
  '3-voyage contract': np.float64(0.0),
  '6-voyage COA': np.float64(3.5),
  '12-voyage COA': np.float64(3.5)}}